# Demo 09 : Control

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · interrupt-on

interrupt_on pauses before write_file and waits for a decision. Approving resumes the same thread. The write does not run while the interrupt is pending.

In [2]:
import json
import sqlite3
import threading
import time
from pathlib import Path

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend, StoreBackend, CompositeBackend
from deepagents.backends.utils import create_file_data
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1800, max_retries=0)
metrics = []
LIMIT = {"recursion_limit": 24}
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}
WORKSPACE = (Path.cwd() / "workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)


def ask(text):
    return {"messages": [HumanMessage(text)]}


def answer(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


async def measured_async(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return await operation()
    finally:
        row = dict(label=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, malformed=counter.invalid, errors=counter.errors,
                   seconds=round(time.perf_counter() - start, 1))
        metrics.append(row)
        print("metrics :", json.dumps(row))


def show(result):
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer :", answer(result))


@tool
def part_stock(part: str) -> str:
    """Look up stock, storage location and name for a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Find the manual's meaning, first action and replacement condition for a bus error code."""
    return "\n".join(f"[{name}]\n{line}" for name, text in PAGES.items()
                     for line in text.splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "Unknown error code."


STOCK_TASK = "Look up CE-CF-02 stock and location. Answer briefly from the tool."
MANUAL_TASK = "Look up F-CE-110. Give the first action and replacement condition, citing the file."
QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition? Be brief.")

In [3]:
gated = create_deep_agent(model=llm, interrupt_on={"write_file": True}, checkpointer=InMemorySaver(),
    system_prompt="Call write_file once for /note.md with the text fan stock checked. Then stop. Do not use another tool.")
config = {**LIMIT, "configurable": {"thread_id": "approve-write"}}
paused = measured("interrupt", lambda: gated.invoke(ask("Write the note."), config, version="v2"))
print("pending :", len(paused.interrupts))
print("actions :", paused.interrupts[0].value["action_requests"])
print("files while paused :", sorted(gated.get_state(config).values.get("files", {})))
resumed = measured("approved", lambda: gated.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config, version="v2"))
print("pending after approve :", len(resumed.interrupts))
show(resumed.value)
print("files :", sorted(resumed.value.get("files", {})))

metrics : {"label": "interrupt", "calls": 1, "input_tokens": 1997, "output_tokens": 27, "malformed": 0, "errors": 0, "seconds": 4.0}
pending : 1
actions : [{'name': 'write_file', 'args': {'file_path': '/note.md', 'content': 'fan stock checked'}, 'description': "Tool execution requires approval\n\nTool: write_file\nArgs: {'file_path': '/note.md', 'content': 'fan stock checked'}"}]
files while paused : []


metrics : {"label": "approved", "calls": 1, "input_tokens": 2040, "output_tokens": 19, "malformed": 0, "errors": 0, "seconds": 2.4}
pending after approve : 0
tools : ['write_file']
answer : Wrote `/note.md` with the text: `fan stock checked`
files : ['/note.md']


## b · summarisation

This model has no profile, so the harness default waits for 170000 tokens and then keeps 6 messages. We replace that middleware with a 3-message trigger so one short lookup stores the older turn.

In [4]:
from deepagents.backends.utils import file_data_to_string
from deepagents.middleware.summarization import SummarizationMiddleware, compute_summarization_defaults

print("defaults :", compute_summarization_defaults(llm))
backend = StateBackend()
tight = SummarizationMiddleware(model=llm, backend=backend, trigger=("messages", 3),
    keep=("messages", 2), truncate_args_settings=None)
compact = create_deep_agent(model=llm, tools=[part_stock], backend=backend, middleware=[tight],
    system_prompt="Use part_stock once, then answer in one sentence. Do not write files.")
result = measured("summarise", lambda: compact.invoke(ask(STOCK_TASK), LIMIT))
show(result)
print("files :", sorted(result.get("files", {})))
for path, payload in sorted(result.get("files", {}).items()):
    if "conversation_history" in path:
        print(path, ":", file_data_to_string(payload)[:400])

defaults : {'trigger': ('tokens', 170000), 'keep': ('messages', 6), 'truncate_args_settings': {'trigger': ('messages', 20), 'keep': ('messages', 20)}}


metrics : {"label": "summarise", "calls": 3, "input_tokens": 4880, "output_tokens": 124, "malformed": 0, "errors": 0, "seconds": 7.8}
tools : ['part_stock']
answer : CE-CF-02 (traction cooling fan) has 6 in stock at Rietpoort.
files : ['/conversation_history/session_74ee0533c4004286bc6e09dd283311d5.md']
/conversation_history/session_74ee0533c4004286bc6e09dd283311d5.md : ## Summarized at 2026-10-26T22:04:24.189207+00:00

<message type="human">Look up CE-CF-02 stock and location. Answer briefly from the tool.</message>




## Things to try

Optional : we can change the bus task and compare the answer, calls and tokens.